# Lahore POIs (Overture / OSM Fallback)
## Export: point GeoJSON of Lahore POIs


### 0. Initialize Imports


In [5]:
import os
import geopandas as gpd

### 1. Parameters


In [6]:
BOUNDARY_PATH = "../../data/Lahore UCs/Lahore UC.shp"
OUT_GEOJSON = "./data/pois_lahore.geojson"

POI_SOURCE = "overture"
OVERTURE_RELEASE = "latest"
OVERTURE_S3_ROOT = "s3://overturemaps-us-west-2/release"

OVERPASS_URLS = [
    "https://overpass-api.de/api/interpreter",
    "https://overpass.kumi.systems/api/interpreter",
    "https://overpass.nchc.org.tw/api/interpreter",
]
SIMPLIFY_TOL_DEG = 0.002


### 2. Load Lahore Boundary and Define Fetchers


In [7]:
boundary = gpd.read_file(BOUNDARY_PATH)
if boundary.crs is None or boundary.crs.to_epsg() != 4326:
    boundary = boundary.to_crs(4326)
lahore_boundary = boundary.geometry.unary_union


def resolve_overture_release(con, configured_release):
    if configured_release != "latest":
        return configured_release
    latest = con.execute("SELECT latest FROM 'https://stac.overturemaps.org/catalog.json'").fetchone()
    if not latest or not latest[0]:
        raise RuntimeError("Could not resolve the latest Overture release from the STAC catalog.")
    return latest[0]


def fetch_overture_pois(boundary_gdf, boundary_polygon):
    import duckdb

    minx, miny, maxx, maxy = boundary_gdf.total_bounds

    con = duckdb.connect()
    con.execute("INSTALL httpfs")
    con.execute("LOAD httpfs")
    con.execute("INSTALL spatial")
    con.execute("LOAD spatial")
    con.execute("SET s3_region='us-west-2'")

    overture_release = resolve_overture_release(con, OVERTURE_RELEASE)
    overture_path = f"{OVERTURE_S3_ROOT}/{overture_release}/theme=places/type=place/*.parquet"
    print(f"Using Overture release: {overture_release}")

    schema = con.execute(
        f"DESCRIBE SELECT * FROM read_parquet('{overture_path}') LIMIT 1"
    ).fetchall()
    cols = [row[0] for row in schema]

    if "bbox" in cols:
        bbox_pred = (
            f"bbox.xmin <= {maxx} AND bbox.xmax >= {minx} "
            f"AND bbox.ymin <= {maxy} AND bbox.ymax >= {miny}"
        )
    else:
        bbox_pred = "1=1"

    query = (
        "SELECT id, names, categories, taxonomy, basic_category, confidence, operating_status, "
        "ST_AsWKB(geometry) AS geometry_wkb "
        f"FROM read_parquet('{overture_path}', filename=true, hive_partitioning=1) "
        f"WHERE {bbox_pred}"
    )

    df = con.execute(query).df()
    con.close()

    geom_raw = df["geometry_wkb"]
    geom_raw = geom_raw.apply(lambda value: bytes(value) if isinstance(value, bytearray) else value)
    mask = geom_raw.apply(lambda value: isinstance(value, (bytes, bytearray)))
    df = df.loc[mask].copy()
    geom_raw = geom_raw[mask]

    df["geometry"] = gpd.GeoSeries.from_wkb(geom_raw)
    df = df.drop(columns=["geometry_wkb"])
    gdf = gpd.GeoDataFrame(df, geometry="geometry", crs=4326)
    return gdf[gdf.geometry.within(boundary_polygon)].copy()


def fetch_osmnx_pois(boundary_polygon):
    import osmnx as ox

    poly = boundary_polygon
    try:
        poly = poly.buffer(0).simplify(SIMPLIFY_TOL_DEG)
    except Exception:
        pass

    tags = {
        "amenity": True,
        "shop": True,
        "tourism": True,
        "leisure": True,
        "office": True,
        "craft": True,
        "man_made": True,
        "public_transport": True,
        "railway": True,
    }

    last_error = None
    for url in OVERPASS_URLS:
        try:
            ox.settings.overpass_url = url
            if hasattr(ox, "features_from_polygon"):
                gdf = ox.features_from_polygon(poly, tags)
            else:
                gdf = ox.geometries_from_polygon(poly, tags)

            if gdf is not None and len(gdf) > 0:
                gdf = gdf.reset_index(drop=True)
                gdf["geometry"] = gdf.geometry.centroid
                gdf = gdf[~gdf.geometry.is_empty].copy()
                return gdf.set_crs(4326, allow_override=True) if gdf.crs is None else gdf.to_crs(4326)
        except Exception as exc:
            last_error = exc
            print(f"Overpass failed: {url} -> {exc}")

    if last_error is not None:
        print(f"OSM fallback returned no data. Last error: {last_error}")

    return gpd.GeoDataFrame(columns=["geometry"], geometry="geometry", crs="EPSG:4326")


/var/folders/80/08nsjy_s5f3brjxw1crm6mbh0000gn/T/ipykernel_45614/1043897972.py:4: DeprecationWarning: The 'unary_union' attribute is deprecated, use the 'union_all()' method instead.
  lahore_boundary = boundary.geometry.unary_union


### 3. Fetch and Export GeoJSON


In [8]:
os.makedirs("./data", exist_ok=True)

if POI_SOURCE == "overture":
    pois = fetch_overture_pois(boundary, lahore_boundary)
elif POI_SOURCE == "osmnx":
    pois = fetch_osmnx_pois(lahore_boundary)
else:
    raise ValueError("Unknown POI_SOURCE")

print(f"POIs fetched: {len(pois)}")
pois.to_file(OUT_GEOJSON, driver="GeoJSON")
print(f"Saved: {OUT_GEOJSON}")


Using Overture release: 2026-03-18.0


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

POIs fetched: 36645
Saved: ./data/pois_lahore.geojson
